In [ ]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib

# add path to project root

import sys
PROJECT_ROOT = Path.cwd()

if "notebooks" in PROJECT_ROOT.parts:
    PROJECT_ROOT = PROJECT_ROOT.parent
    
sys.path.append(str(PROJECT_ROOT))

    
from src.preprocess.gaming_text_preprocessor import GamingTextPreprocessor
from src.preprocess.gaming_text_labeler import GamingTextLabeler
from src.ensemble.ensemble import Ensemble
from src.model.game_model_collection import GamingModelCollection
from src.model.social_media_collection import SocialMediaModelCollection
from src.model.bert_collection import BertToxicityModelCollection
from src.ensemble.score import ClassificationMetrics


CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'model_dir': PROJECT_ROOT / 'models',
    'text_col': 'message',
    'label_col': 'label'
}


print(PROJECT_ROOT)

# social media
social_paths = list((CONFIG['model_dir'] / 'binary' ).glob('social_media_*.joblib'))

scaler_path = CONFIG['model_dir'] / 'helper' / 'social_media_max_abs_scaler.joblib'
nb_tfidf_path = CONFIG['model_dir'] / 'helper' / 'social_media_nb_tfidf.joblib'


# gaming
model_dir = CONFIG['model_dir'] / "binary"
gaming_model_paths = list(model_dir.glob("gaming_all_*.joblib"))



/Users/paolacalle/Desktop/NYU/semesters/spring-2026/ML/hw/gaming-toxicity-detection


In [13]:
np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

# Load Data

In [14]:
# load WOT train data
d = CONFIG['data_dir']

wot_train = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train["data_source"] = "wot"

# load DOTA training data
dota_train = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train["data_source"] = "dota"


# combine datasets
train = pd.concat([wot_train, dota_train], ignore_index=True)

In [15]:
wot_val = pd.read_parquet(d / 'wot' / 'x_validation.parquet')
wot_val["data_source"] = "wot"
dota_val = pd.read_parquet(d / 'dota' / 'x_validation.parquet')
dota_val["data_source"] = "dota"

# combine holdout datasets
val = pd.concat([wot_val, dota_val], ignore_index=True)

# Prepare Objects

In [16]:
# instantiate preprocessor and labeler
gaming_preprocessor = GamingTextPreprocessor()
gaming_labeler = GamingTextLabeler()
metrics = ClassificationMetrics()

gaming_preprocessor.fit(train, text_column=tc)

In [17]:
# convert dota labels 
train = gaming_labeler.convert_binary(
    train, 
    label_column=lc, 
    output_column='label_binary'
)
val = gaming_labeler.convert_binary(
    val, 
    label_column=lc, 
    output_column='label_binary'
)


In [18]:
gaming_model_collection = GamingModelCollection(
    model_joblibs = gaming_model_paths
)

In [19]:
gaming_model_collection.classifiers.keys()

dict_keys(['wot_Logistic_Regression', 'wot_LinearSVC', 'wot_Logistic_Regression_+_Calibrated(isotonic,_ensemble=True)', 'wot_LinearSVC_+_Calibrated(isotonic,_ensemble=True)', 'dota_Logistic_Regression', 'dota_LinearSVC', 'dota_Logistic_Regression_+_Calibrated(sigmoid,_ensemble=False)', 'dota_LinearSVC_+_Calibrated(sigmoid,_ensemble=True)', 'combined_Logistic_Regression', 'combined_LinearSVC', 'combined_Logistic_Regression_+_Calibrated(isotonic,_ensemble=True)', 'combined_LinearSVC_+_Calibrated(sigmoid,_ensemble=False)'])

In [20]:
social_media_collection = SocialMediaModelCollection(
    model_joblibs=social_paths,
    scaler_path=scaler_path,
    nb_tfidf_path=nb_tfidf_path,
)

In [21]:
social_media_collection.classifiers

{PosixPath('/Users/paolacalle/Desktop/NYU/semesters/spring-2026/ML/hw/gaming-toxicity-detection/models/binary/social_media_ComplementNB_model.joblib'): Pipeline(steps=[('clf', ComplementNB(alpha=1.2006196407511314))]),
 PosixPath('/Users/paolacalle/Desktop/NYU/semesters/spring-2026/ML/hw/gaming-toxicity-detection/models/binary/social_media_LinearSVC_(Optuna)_model.joblib'): Pipeline(steps=[('clf',
                  LinearSVC(C=0.7813882258601701, class_weight='balanced',
                            dual=False, max_iter=5000, penalty='l1',
                            random_state=42))]),
 PosixPath('/Users/paolacalle/Desktop/NYU/semesters/spring-2026/ML/hw/gaming-toxicity-detection/models/binary/social_media_LogisticRegressionCV_model.joblib'): Pipeline(steps=[('clf',
                  LogisticRegressionCV(class_weight='balanced',
                                       cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                                       max_iter=1000, n_

In [22]:
bert_binary_collection = BertToxicityModelCollection(
    model_names=["dota_binary", "jigsaw_binary", "wot_binary"]
)

Loading dota_binary from jforward/bert-toxicity/dota_binary_model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10232.00it/s]


Loading jigsaw_binary from jforward/bert-toxicity/jigsaw_binary_model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 16341.44it/s]


Loading wot_binary from jforward/bert-toxicity/wot_binary_model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 15163.95it/s]


In [23]:
social_gaming_ensemble = Ensemble(
    model_collections=[social_media_collection, gaming_model_collection, bert_binary_collection]
)

# Run Simple Ensemble 

In [13]:
pred_train = social_gaming_ensemble.predict_simple_majority(train[tc])
metrics.metrics(train['label_binary'], pred_train)

Predicting with SimpleMajority...


{'CV Macro F1': 0.9670465072461059,
 'CV Weighted F1': 0.9760777197391513,
 'Accuracy': 0.9760552628149031,
 'Coverage': np.float64(1.0),
 'Precision': 0.9472291407222914,
 'FPR': np.float64(0.016565271567836985),
 'FNR': np.float64(0.04758178118641415),
 'TPR': np.float64(0.9524182188135859),
 'TNR': np.float64(0.9834347284321631)}

In [14]:
pred_val = social_gaming_ensemble.predict_simple_majority(val[tc])
metrics.metrics(val['label_binary'], pred_val)

Predicting with SimpleMajority...


{'CV Macro F1': 0.8439607663661371,
 'CV Weighted F1': 0.8947746516267456,
 'Accuracy': 0.8956965290806754,
 'Coverage': np.float64(1.0),
 'Precision': 0.7728045325779037,
 'FPR': np.float64(0.060070406711107784),
 'FNR': np.float64(0.263697705802969),
 'TPR': np.float64(0.7363022941970311),
 'TNR': np.float64(0.9399295932888923)}

# Run Weighted Majority Ensemble 

In [16]:
weights = social_gaming_ensemble.fit_weighted_majority(
    X_val=train[tc],
    y_val=train['label_binary'],
    score_func=metrics.score,
    n_trials=500,
    random_state=CONFIG['seed']
)[0]

In [17]:
weights

array([0.04100003, 0.00253944, 0.02768951, 0.05121442, 0.03849329,
       0.20625597, 0.03040364, 0.1493109 , 0.13375895, 0.02159138,
       0.08483262, 0.16917333, 0.01888329, 0.00765747, 0.01719574])

In [18]:
pred_train = social_gaming_ensemble.predict_weighted_majority(
    X=train[tc],
    weights=weights
)
metrics.metrics(train['label_binary'], pred_train)

{'CV Macro F1': 0.967930764580603,
 'CV Weighted F1': 0.9767055538600615,
 'Accuracy': 0.9766697078593107,
 'Coverage': np.float64(1.0),
 'Precision': 0.9469479562553323,
 'FPR': np.float64(0.016711866891446162),
 'FNR': np.float64(0.04452966035373298),
 'TPR': np.float64(0.955470339646267),
 'TNR': np.float64(0.9832881331085538)}

In [19]:
pred_val = social_gaming_ensemble.predict_weighted_majority(
    X=val[tc],
    weights=weights
)
metrics.metrics(val['label_binary'], pred_val)

{'CV Macro F1': 0.8415124837888408,
 'CV Weighted F1': 0.8931499236231444,
 'Accuracy': 0.8941135084427767,
 'Coverage': np.float64(1.0),
 'Precision': 0.7693617021276595,
 'FPR': np.float64(0.06089431503258183),
 'FNR': np.float64(0.2680161943319838),
 'TPR': np.float64(0.7319838056680162),
 'TNR': np.float64(0.9391056849674182)}

# Run Weighted Confidence Majority Ensemble 

In [20]:
res = social_gaming_ensemble.fit_weighted_confidence_majority(
    X_val=train[tc],
    y_val=train['label_binary'],
    score_func=metrics.score,
    n_trials=100,
    random_state=CONFIG['seed']
)

In [21]:
pred_train = social_gaming_ensemble.predict_weighted_confidence_majority(
    X=train[tc],
    weights=res[0]
)
metrics.metrics(train['label_binary'], pred_train)

{'CV Macro F1': 0.9672567694355423,
 'CV Weighted F1': 0.976228440246519,
 'Accuracy': 0.9762042191893049,
 'Coverage': np.float64(1.0),
 'Precision': 0.9473315699393186,
 'FPR': np.float64(0.016540839013902124),
 'FNR': np.float64(0.047033964626702146),
 'TPR': np.float64(0.9529660353732978),
 'TNR': np.float64(0.9834591609860979)}

In [22]:
pred_val = social_gaming_ensemble.predict_weighted_confidence_majority(
    X=val[tc],
    weights=res[0]
)
metrics.metrics(val['label_binary'], pred_val)

{'CV Macro F1': 0.8426699101229502,
 'CV Weighted F1': 0.8940397972623165,
 'Accuracy': 0.8951102251407129,
 'Coverage': np.float64(1.0),
 'Precision': 0.7734018264840182,
 'FPR': np.float64(0.05947120065912666),
 'FNR': np.float64(0.26855600539811064),
 'TPR': np.float64(0.7314439946018894),
 'TNR': np.float64(0.9405287993408733)}

In [23]:
thresholds = np.arange(0.5, 0.9, 0.1)

res = social_gaming_ensemble.fit_weighted_confidence_majority(
    X_val=train[tc],
    y_val=train['label_binary'],
    score_func=metrics.score,
    n_trials=50,
    random_state=CONFIG['seed'],
    thresholds=thresholds
)

In [24]:
pred_train = social_gaming_ensemble.predict_weighted_confidence_majority(
    X=train[tc],
    weights=res[0]
)
metrics.metrics(train['label_binary'], pred_train)

{'CV Macro F1': 0.9672567694355423,
 'CV Weighted F1': 0.976228440246519,
 'Accuracy': 0.9762042191893049,
 'Coverage': np.float64(1.0),
 'Precision': 0.9473315699393186,
 'FPR': np.float64(0.016540839013902124),
 'FNR': np.float64(0.047033964626702146),
 'TPR': np.float64(0.9529660353732978),
 'TNR': np.float64(0.9834591609860979)}

In [25]:
pred_val = social_gaming_ensemble.predict_weighted_confidence_majority(
    X=val[tc],
    weights=res[0]
)
metrics.metrics(val['label_binary'], pred_val)

{'CV Macro F1': 0.8426699101229502,
 'CV Weighted F1': 0.8940397972623165,
 'Accuracy': 0.8951102251407129,
 'Coverage': np.float64(1.0),
 'Precision': 0.7734018264840182,
 'FPR': np.float64(0.05947120065912666),
 'FNR': np.float64(0.26855600539811064),
 'TPR': np.float64(0.7314439946018894),
 'TNR': np.float64(0.9405287993408733)}